# CDEA | Main + Scale analysis

**Research analysis:** paired task completion, 3B-9B model-size follow-up, task strata, mechanism indicators, efficiency, and task-cluster sensitivity intervals.

The analysis recalculates outcomes from supplied Main and Scale ZIPs. These are **separate studies** and must not be pooled as independent observations. Source files and SHA-256 fingerprints are retained in the exports.

## 1. Datasets, inclusion rules, and statistical methods

Place the original ZIP files beside the notebook (or in Colab's Files pane). Set the paths only if multiple candidate archives are available. Invalid and unstarted pairs remain in the sample counts and are excluded only from paired success denominators.

In [ ]:
from pathlib import Path
import io, json, hashlib, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from IPython.display import display

MAIN_ZIP = ""     # e.g. "/content/main_dataset.zip"
SCALE_ZIP = ""    # e.g. "/content/scale_reference_v4.zip"
SEED, BOOTSTRAPS = 233, 3000
OUT = Path("analysis_outputs_compact")
OUT.mkdir(parents=True, exist_ok=True)


def locate(protocol, explicit=""):
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    candidates = []
    for folder in [Path.cwd(), Path("/content"), Path("/mnt/data")]:
        if folder.is_dir():
            candidates.extend(folder.glob("*.zip"))
    found = []
    for path in sorted(set(p.resolve() for p in candidates)):
        try:
            with zipfile.ZipFile(path) as z:
                for name in z.namelist():
                    if name.endswith("manifest.json") and name.count("/") <= 1:
                        if json.loads(z.read(name)).get("protocol") == protocol:
                            found.append(path)
                        break
        except (OSError, ValueError, KeyError, zipfile.BadZipFile):
            continue
    if len(found) > 1:
        raise ValueError(f"Multiple {protocol} archives found; specify MAIN_ZIP or SCALE_ZIP: {found}")
    return found[0] if found else None


def boolean(series):
    return series.astype("string").str.strip().str.lower().map(
        {"true":True,"false":False,"1":True,"0":False})


def load_study(path, protocol):
    if path is None:
        return None
    with zipfile.ZipFile(path) as z:
        manifests = [x for x in z.namelist() if x.endswith("manifest.json") and x.count("/") <= 1]
        if len(manifests) != 1:
            raise ValueError(f"Expected one manifest in {path.name}")
        prefix = manifests[0][:-len("manifest.json")]
        manifest = json.loads(z.read(prefix + "manifest.json"))
        if manifest.get("protocol") != protocol:
            raise ValueError(f"Wrong protocol in {path.name}: {manifest.get('protocol')}")
        pairs = pd.read_csv(io.BytesIO(z.read(prefix + "pairs.csv")))
        results = pd.read_csv(io.BytesIO(z.read(prefix + "results.csv")))
    for key in ("paired_valid", "baseline_success", "proposed_success"):
        pairs[key] = boolean(pairs[key])
    for key in ("paired_analysis_success", "official_success_raw", "episode_valid"):
        if key in results:
            results[key] = boolean(results[key])
    if len(pairs) != manifest["planned_pairs"] or len(results) > manifest["planned_episodes"]:
        raise ValueError("Scheduled denominator or episode count mismatch")
    if pairs.pair_id.duplicated().any() or results.duplicated(["pair_id","method"]).any():
        raise ValueError("Duplicate pair or episode slot")
    if set(pairs.pair_id) != {p["pair_id"] for p in manifest["pairs"]}:
        raise ValueError("Manifest and pair roster differ")
    if pairs.paired_valid.isna().any() or not set(results.method).issubset({"Baseline","Proposed"}):
        raise ValueError("Missing pair validity or unexpected method")
    valid = pairs.loc[pairs.paired_valid.eq(True)].copy()
    if valid[["baseline_success","proposed_success"]].isna().any().any():
        raise ValueError("Valid pair lacks an endpoint")
    slots = results.loc[results.pair_id.isin(valid.pair_id)]
    counts = slots.groupby("pair_id").method.nunique()
    if len(counts) != len(valid) or not counts.eq(2).all():
        raise ValueError("Valid pairs must have exactly two episode records")
    if {"official_success_raw", "paired_analysis_success"}.issubset(slots.columns):
        if not slots.official_success_raw.eq(slots.paired_analysis_success).all():
            raise ValueError("Episode recorded official outcomes differ from paired endpoint values")
    if "analysis_included" in slots.columns:
        if not boolean(slots.analysis_included).eq(True).all():
            raise ValueError("Valid pairs contain episodes excluded from paired analysis")
    if "paired_analysis_success" in slots.columns:
        observed = slots.pivot(index="pair_id", columns="method", values="paired_analysis_success")
        check = valid.set_index("pair_id")
        if not (observed.loc[check.index, "Baseline"].eq(check.baseline_success).all()
                and observed.loc[check.index, "Proposed"].eq(check.proposed_success).all()):
            raise ValueError("Pair endpoints disagree with episode endpoints")
    valid["delta"] = valid.proposed_success.astype(int) - valid.baseline_success.astype(int)
    valid["task_key"] = valid.task_group.astype(str) + "/" + valid.task.astype(str)
    return {"manifest":manifest,"pairs":pairs,"valid":valid,"results":results,
            "path":path,"sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
            "n_invalid":int((~pairs.paired_valid.fillna(False)).sum()),
            "n_not_started":int(pairs.pair_status.eq("not_started").sum())}


def task_ci(df, draws=BOOTSTRAPS, seed=SEED):
    if df.empty: return (np.nan, np.nan)
    grouped = df.groupby("task_key").delta.agg(["sum","count"]).to_numpy()
    if len(grouped) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    ix = rng.integers(0, len(grouped), size=(draws, len(grouped)))
    chosen = grouped[ix]
    sampled_pp = 100 * chosen[:,:,0].sum(axis=1) / chosen[:,:,1].sum(axis=1)
    return tuple(np.quantile(sampled_pp, [0.025, 0.975]))


def stats(df, cluster=True):
    n = len(df)
    if not n: return dict(N=0, baseline=0, cdea=0, baseline_pct=np.nan,
                          cdea_pct=np.nan, delta_pp=np.nan, proposed_only=0,
                          baseline_only=0, both_success=0, neither_success=0,
                          relative_gain_pct=np.nan, mcnemar_p=np.nan, ci_low=np.nan, ci_high=np.nan)
    b, p = df.baseline_success.astype(bool), df.proposed_success.astype(bool)
    po, bo = int((~b & p).sum()), int((b & ~p).sum())
    both, neither = int((b & p).sum()), int((~b & ~p).sum())
    low, high = task_ci(df) if cluster else (np.nan, np.nan)
    return dict(N=n, baseline=int(b.sum()), cdea=int(p.sum()),
                baseline_pct=100*b.mean(), cdea_pct=100*p.mean(),
                delta_pp=100*(p.mean()-b.mean()), proposed_only=po, baseline_only=bo,
                both_success=both, neither_success=neither,
                relative_gain_pct=(100*(int(p.sum())/int(b.sum()) - 1) if b.any() else np.nan),
                mcnemar_p=binomtest(min(po,bo),po+bo).pvalue if po+bo else 1.0,
                ci_low=low, ci_high=high)

main = load_study(locate("cdea_main_v1", MAIN_ZIP), "cdea_main_v1")
scale = load_study(locate("cdea_scale_v1", SCALE_ZIP), "cdea_scale_v1")
if main is None and scale is None:
    raise FileNotFoundError("No Main or Scale ZIP detected. Upload files or set MAIN_ZIP / SCALE_ZIP.")

population = pd.DataFrame([{"study":name,"archive":s["path"].name,
    "planned_pairs":len(s["pairs"]),"valid_pairs":len(s["valid"]),
    "invalid_pairs":int((~s["pairs"].paired_valid).sum())-s["n_not_started"],
    "not_started_pairs":s["n_not_started"],"episode_records":len(s["results"])}
    for name,s in [("Main",main),("Scale",scale)] if s is not None])
display(population)

## 2. Matched-pair outcomes and statistical uncertainty

Success rate is official task completion **as recorded** per valid Baseline/Proposed pair. Delta is **CDEA minus Baseline in percentage points**. McNemar exact p-values are exploratory and unadjusted; task-bootstrap intervals resample task IDs as a sensitivity analysis, not as population-wide guarantees.

In [ ]:
rows = []
def add(study, group, df):
    if len(df): rows.append({"study":study,"group":group,**stats(df)})
if main is not None:
    add("Main", "Primary: bank information", main["valid"].loc[main["valid"].stratum.eq("bank_information")])
    add("Main", "All tasks", main["valid"])
if scale is not None:
    add("Scale", "All tasks", scale["valid"])
    for size, subset in scale["valid"].groupby("size"):
        add("Scale", f"Model {size}", subset)
overview = pd.DataFrame(rows)
summary_display = overview.round(3).copy()
summary_display["mcnemar_p"] = overview.mcnemar_p.map(
    lambda p: "<0.001" if p < 0.001 else f"{p:.3f}")
display(summary_display)
print("Interpretation: an effect is positive when CDEA exceeds Baseline; correlated tasks/model runs constrain inference.")

# Matched outcomes, including the discordant pairs driving McNemar's test.
paired_outcomes = overview[["study", "group", "N", "both_success", "proposed_only",
    "baseline_only", "neither_success", "relative_gain_pct", "mcnemar_p", "ci_low", "ci_high"]].copy()
paired_display = paired_outcomes.round(3).copy()
paired_display["mcnemar_p"] = paired_outcomes.mcnemar_p.map(
    lambda p: "<0.001" if p < 0.001 else f"{p:.3f}")
display(paired_display)
print("Two-sided McNemar p-values are exploratory; intervals resample fixed task IDs.")


## 3. Checkpoint, task-category, and model-size heterogeneity

Within-family small versus large is calculated on **common valid task IDs** only. The xLAM 3B/8B pair is not a clean parameter-size contrast because the backbones differ.

In [ ]:
family_size = pd.DataFrame()
interactions = pd.DataFrame()
strata = pd.DataFrame()
if scale is not None:
    frame = scale["valid"]
    family_size = pd.DataFrame([
        {"family":f,"size":s,**stats(grp)} for (f,s),grp in frame.groupby(["family","size"])
    ]).sort_values(["family","size"]).reset_index(drop=True)
    interaction_rows = []
    for family, group in frame.groupby("family"):
        paired = group.pivot_table(index="task_key", columns="size", values="delta", aggfunc="first").dropna()
        if len(paired):
            d = (paired["small"] - paired["large"]).to_numpy(dtype=float)
            rng = np.random.default_rng(SEED)
            bs = 100 * d[rng.integers(0,len(d),(BOOTSTRAPS,len(d)))].mean(axis=1) if len(d)>1 else []
            ci = np.quantile(bs,[.025,.975]) if len(bs) else [np.nan,np.nan]
            interaction_rows.append({"family":family,"common_tasks":len(d),
                "small_minus_large_pp":100*d.mean(),"ci_low":ci[0],"ci_high":ci[1],
                "backbone_confound":family=="xlam2"})
    interactions = pd.DataFrame(interaction_rows)
    strata = pd.DataFrame([
        {"stratum":s,"size":z,**stats(grp)} for (s,z),grp in frame.groupby(["stratum","size"])
    ]).sort_values(["stratum","size"]).reset_index(drop=True)
    print("Model-family breakdown (32 task IDs per checkpoint):")
    display(family_size[["family","size","N","baseline_pct","cdea_pct","delta_pp","ci_low","ci_high"]].round(2))
    print("Interaction = benefit(small) - benefit(large), among common valid tasks:")
    display(interactions.round(2))
    print("Task-stratum breakdown:")
    display(strata[["stratum","size","N","baseline_pct","cdea_pct","delta_pp"]].round(2))
else:
    print("No Scale ZIP supplied; size comparison is skipped.")

# Restore Main study's per-checkpoint and per-task-category reports.
main_models = pd.DataFrame()
main_strata = pd.DataFrame()
if main is not None:
    m = main["valid"]
    main_models = pd.DataFrame([{"model_label":label, **stats(group)}
        for label, group in m.groupby("model_label")]).sort_values("model_label")
    main_strata = pd.DataFrame([{"stratum":label, **stats(group)}
        for label, group in m.groupby("stratum")]).sort_values("stratum")
    print("Main — comparison by checkpoint:")
    display(main_models[["model_label", "N", "baseline_pct", "cdea_pct", "delta_pp",
                         "proposed_only", "baseline_only"]].round(2))
    print("Main — comparison by task category:")
    display(main_strata[["stratum", "N", "baseline_pct", "cdea_pct", "delta_pp",
                         "proposed_only", "baseline_only"]].round(2))

scale_models = pd.DataFrame()
if scale is not None:
    scale_models = pd.DataFrame([{"model_label":label, "family":group.family.iloc[0],
        "size":group["size"].iloc[0], **stats(group)}
        for label,group in scale["valid"].groupby("model_label")]).sort_values("model_label")
    print("Scale — all eight checkpoints:")
    display(scale_models[["model_label", "size", "N", "baseline_pct", "cdea_pct",
                          "delta_pp", "proposed_only", "baseline_only"]].round(2))


## 4. CDEA mechanism, efficiency, and failure patterns

Probe opportunities and policy divergences measure **where the policies could differ**, not whether information was correctly acquired. Efficiency is descriptive, not a controlled estimate of model compute cost.

In [ ]:
mechanism_rows, efficiency_rows = [], []
for name, study in [("Main",main),("Scale",scale)]:
    if study is None: continue
    v = study["valid"]
    # Prefer pair-level diagnostics to avoid double counting two arms.
    for measure in ("comparable_probe_decisions", "policy_action_divergences"):
        if measure in v:
            series = pd.to_numeric(v[measure],errors="coerce").fillna(0)
        else:
            proposed = study["results"].loc[(study["results"].method == "Proposed") &
                (study["results"].pair_id.isin(v.pair_id))]
            series = pd.to_numeric(proposed.get(measure,pd.Series(dtype=float)),errors="coerce").fillna(0)
        mechanism_rows.append({"study":name,"metric":measure,"events":int(series.sum()),
            "pairs_with_event":int(series.gt(0).sum()),"valid_pairs":len(v)})
    episodes = study["results"].loc[study["results"].pair_id.isin(v.pair_id)].copy()
    for method, group in episodes.groupby("method"):
        record = {"study":name,"method":method,"episodes":len(group)}
        for metric in ("steps", "model_seconds", "total_seconds", "repair_requests"):
            if metric in group:
                value = pd.to_numeric(group[metric],errors="coerce")
                record[f"median_{metric}"] = float(value.median())
        efficiency_rows.append(record)
mechanism = pd.DataFrame(mechanism_rows)
efficiency = pd.DataFrame(efficiency_rows)
display(mechanism)
display(efficiency.round(2))
print("Caution: behavior divergence is not direct evidence that CDEA obtained correct new information.")

# Outcome association with genuine ranking-opportunity diagnostics.
# Descriptive associations are NOT randomized within the diverged subset.
mechanism_effect_rows = []
for name,study in (("Main",main),("Scale",scale)):
    if study is None: continue
    df = study["valid"].copy()
    if "policy_action_divergences" in df:
        df["divergence_observed"] = pd.to_numeric(df["policy_action_divergences"], errors="coerce").fillna(0).gt(0)
        for observed,group in df.groupby("divergence_observed"):
            mechanism_effect_rows.append({"study":name, "action_divergence":bool(observed),
                                          **stats(group)})
mechanism_effect = pd.DataFrame(mechanism_effect_rows)
print("Matched outcomes among pairs with / without recorded ranking divergence:")
display(mechanism_effect[["study","action_divergence","N","proposed_only",
    "baseline_only","delta_pp"]].round(2)) if not mechanism_effect.empty else None

# Inspect information-acquisition evidence and stop reasons within stored episode traces.
# Count missing trace files explicitly; absence of evidence must not be mistaken for zero.
trace_rows, failure_rows = [], []
for name,study in (("Main",main),("Scale",scale)):
    if study is None: continue
    examined = missing = steps = grounded = nonunknown = chosen_probe = 0
    with zipfile.ZipFile(study["path"]) as z:
        roots = [n[:-len("manifest.json")] for n in z.namelist()
                 if n.endswith("manifest.json") and n.count("/") <= 1]
        prefix = roots[0]
        slots = study["results"].loc[study["results"].pair_id.isin(study["valid"].pair_id)]
        for _,row in slots.iterrows():
            default_path = f'episodes/{row["pair_id"]}/{row["method"]}/episode/episode.json'
            artifact = row.get("episode_artifact", default_path)
            artifact = artifact if isinstance(artifact,str) and artifact.endswith("episode.json") else default_path
            entry = prefix + artifact
            if entry not in z.namelist():
                missing += 1
                continue
            ep = json.loads(z.read(entry))
            examined += 1
            trace = ep.get("trace", [])
            if not isinstance(trace, list):
                missing += 1
                continue
            steps += len(trace)
            for item in trace:
                if not isinstance(item,dict): continue
                grounded += len(item.get("grounded_tentative_evidence") or [])
                chosen_probe += int(bool(item.get("selected_probe")))
                ledger = item.get("ledger") or {}
                nonunknown += int(any(isinstance(v,dict) and v.get("state") in ("SATISFIED","REFUTED")
                                      for v in ledger.values()))
            if not bool(row.get("paired_analysis_success", row.get("official_success_raw",False))):
                failure_rows.append({"study":name,"method":row["method"],
                    "end_reason":ep.get("end_reason") or row.get("end_reason") or "unknown"})
    trace_rows.append({"study":name,"episodes_examined":examined,"missing_episode_traces":missing,
        "trace_steps":steps,"selected_probe_actions":chosen_probe,
        "grounded_evidence_entries":grounded,"steps_with_resolved_constraints":nonunknown})
trace_indicators = pd.DataFrame(trace_rows)
failures = (pd.DataFrame(failure_rows).groupby(["study","method","end_reason"],as_index=False)
            .size().rename(columns={"size":"episodes"}) if failure_rows else pd.DataFrame())
print("Trace-level indicators (recorded step diagnostics):")
display(trace_indicators)
print("Unsuccessful episode end reasons:")
display(failures if not failures.empty else pd.DataFrame({"note":["No failure reasons recorded"]}))

# Compare resources within the same task × model pair, respecting episode pairing.
paired_efficiency_rows = []
for name,study in (("Main",main),("Scale",scale)):
    if study is None: continue
    ep = study["results"].loc[study["results"].pair_id.isin(study["valid"].pair_id)]
    for metric in ("steps","model_calls","model_seconds","total_seconds"):
        if metric not in ep: continue
        pivot = ep.pivot(index="pair_id",columns="method",values=metric)
        if not {"Baseline","Proposed"}.issubset(pivot): continue
        change = (pd.to_numeric(pivot["Proposed"],errors="coerce") -
                  pd.to_numeric(pivot["Baseline"],errors="coerce")).dropna()
        paired_efficiency_rows.append({"study":name,"metric":metric,
            "paired_episodes":len(change),"median_cdea_minus_baseline":change.median(),
            "mean_cdea_minus_baseline":change.mean()})
paired_efficiency = pd.DataFrame(paired_efficiency_rows)
print("Paired resource-use differences (negative means CDEA used less):")
display(paired_efficiency.round(2))


## 5. Visualizations, reproducible exports, and conclusions

Plots and tables are regenerated from valid Baseline/Proposed pairs. The exported source inventory records input archive names, protocols, and SHA-256 hashes.

In [ ]:
if not overview.empty:
    fig, ax = plt.subplots(figsize=(9,4.4))
    labels = overview["study"] + " | " + overview["group"]
    x = np.arange(len(overview)); w=.36
    ax.bar(x-w/2, overview.baseline_pct,w,label="Baseline")
    ax.bar(x+w/2, overview.cdea_pct,w,label="CDEA")
    ax.set(ylim=(0,105),ylabel="Success rate (%)",title="Matched task completion")
    ax.set_xticks(x,labels,rotation=20,ha="right"); ax.legend(frameon=False)
    ax.grid(axis="y",alpha=.15);ax.set_axisbelow(True)
    fig.tight_layout();fig.savefig(OUT/"01_success.png",dpi=160);plt.show()
if not family_size.empty:
    fig,ax=plt.subplots(figsize=(9,4.2))
    pivot=family_size.pivot(index="family", columns="size", values="delta_pp").sort_index()
    x=np.arange(len(pivot));w=.36
    for pos,size in enumerate(["small","large"]):
        ax.bar(x + (pos-.5)*w,pivot[size],w,label=size)
    ax.set_xticks(x,pivot.index);ax.set(ylabel="CDEA - Baseline (pp)",title="Effect by model family and size")
    ax.axhline(0,color="gray",lw=.8);ax.legend(frameon=False)
    ax.grid(axis="y",alpha=.15);ax.set_axisbelow(True)
    fig.tight_layout();fig.savefig(OUT/"02_size_effect.png",dpi=160);plt.show()

exported_tables = []
for filename, data in {"population.csv":population,"effects.csv":overview,"model_size.csv":family_size,
                       "size_interactions.csv":interactions,"task_strata.csv":strata,
                       "mechanism.csv":mechanism,"efficiency.csv":efficiency}.items():
    if data.empty: continue
    data.to_csv(OUT/filename,index=False)
    exported_tables.append(filename)
meta = {name:{"archive":s["path"].name,"sha256":s["sha256"],
                "protocol":s["manifest"]["protocol"],"valid_pairs":len(s["valid"])}
        for name,s in [("Main",main),("Scale",scale)] if s is not None}
meta["interpretation"]="Paired outcome summaries from supplied archives. Main and Scale are analyzed separately."
(OUT/"sources_and_scope.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")
for name in ["Main","Scale"]:
    matched = overview[(overview.study == name) & (overview.group == "All tasks")]
    if len(matched):
        r = matched.iloc[0]
        print(f"{name}: Baseline {r.baseline:.0f}/{r.N:.0f} ({r.baseline_pct:.2f}%) vs "
              f"CDEA {r.cdea:.0f}/{r.N:.0f} ({r.cdea_pct:.2f}%), delta {r.delta_pp:+.2f} pp")
print("Wrote analysis tables, two figures, and source scope to", OUT.resolve())

if not strata.empty:
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    comparison = strata.pivot(index="stratum",columns="size",values="delta_pp")
    comparison.reindex(columns=["small","large"]).plot(kind="bar",ax=ax,rot=0)
    ax.axhline(0,color="gray",linewidth=.9)
    ax.set(xlabel="",ylabel="CDEA - Baseline (percentage points)",
           title="Recorded effect by task category and model size")
    ax.grid(axis="y",alpha=.15);ax.set_axisbelow(True)
    fig.tight_layout();fig.savefig(OUT/"03_task_category_effect.png",dpi=160);plt.show()

extras = {"paired_outcomes.csv":paired_outcomes,"main_by_model.csv":main_models,
          "main_by_stratum.csv":main_strata,"scale_by_checkpoint.csv":scale_models,
          "mechanism_by_divergence.csv":mechanism_effect,"trace_indicators.csv":trace_indicators,
          "failure_reasons.csv":failures,"paired_efficiency.csv":paired_efficiency}
for filename,table in extras.items():
    if table.empty: continue
    table.to_csv(OUT/filename,index=False)
    exported_tables.append(filename)
print("Exported populated CSV tables:",len(exported_tables),"; figures:",len(list(OUT.glob("*.png"))))
